### **UNIVERSITY OF SURREY**
MSc DATA SCIENCE

DISSERTATION

**Emotion Based Music Recommendation System with Explainable AI**

NAME: MINAL HONALI RAGHUNANDAN


STUDENT ID: 6908107


**Project Overview**


Create a music recommendation system based on emotion analysis by using DEAM dataset, real user responses and browser plugin data

Notebook 1: data preprocessing of DEAM dataset


In [10]:
import os
import glob
import numpy as np
import pandas as pd
from google.colab import drive
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import kagglehub


#### 1. Dataset Acquisition

This notebook loads the DEAM (Database for Emotional Analysis of Music) dataset via
KaggleHub. DEAM provides per-song valence and arousal annotations (continuous, 1–9 scale,
averaged across multiple human raters) alongside pre-extracted openSMILE acoustic
features for each track. This forms the training foundation for the emotion-regression
model; real-world validation against user-logged data (browser plugin) is handled in
Notebook 03/04.

In [2]:
print("first i'm Downloading DEAM Dataset from kagglehub...")

dataset_dir = kagglehub.dataset_download("imsparsh/deam-mediaeval-dataset-emotional-analysis-in-music")
print(f"Dataset cached {dataset_dir}")
OUTPUT_PATH = "/content/processed_data"
os.makedirs(OUTPUT_PATH, exist_ok=True)


first i'm Downloading DEAM Dataset from kagglehub...


100%|██████████| 1.83G/1.83G [00:22<00:00, 87.9MB/s]

Extracting files...


Dataset cached /root/.cache/kagglehub/datasets/imsparsh/deam-mediaeval-dataset-emotional-analysis-in-music/versions/1


In [3]:
import zipfile
UNZIP_PATH = "/content/deam_extraction"
os.makedirs(UNZIP_PATH, exist_ok=True)
zip_files = glob.glob(os.path.join(dataset_dir,"**","*.zip"), recursive=True)
for z in zip_files:
  with zipfile.ZipFile(z, 'r') as zip_ref:
    zip_ref.extractall(UNZIP_PATH)
print(f"Extracted zip archives tp:{UNZIP_PATH}")

Extracted zip archives tp:/content/deam_extraction


DEAM annotations need to be rescaled from 1-9 rating scale to a [-1,1] range with 0 as center. this is to align with the Russel circumplex model of affect, where 0 represents neutral valence/arousal, positive values indicate pleasant and high energy states, and negative values indicate low energy states. This normalisation is useful as it makes the target space interpretable as quadrant coordinates. This is later used to map the browser plugin data which i built, contains Q1-Q4 quadrants.

LOAD AND CLEAN STATIC ANNOTATIONS (TARGETS)

In [4]:
#cleaning static annootations-> targets
# i will now use glob to locate annotation file paths
# clean column names here by removing the spaces, tails and lead in the names
# CELL 4: Clean Static Annotations (Targets)

# Locate static annotations file across both extracted and cached directories
annotation_files = glob.glob(
    os.path.join(dataset_dir, "**", "static_annotations_averaged_songs_1_2000.csv"),
    recursive=True,
)

if not annotation_files:
  raise FileNotFoundError(
      "Could not find static_annotations_averaged_songs_1_2000.csv"
  )

static_annotation = annotation_files[0]
df_targets = pd.read_csv(static_annotation)

df_targets.columns = df_targets.columns.str.strip()
#target columns are song_id, valence_mean, arousal_mean

df_targets = df_targets[["song_id", "valence_mean", "arousal_mean"]].dropna().copy()

#DEAM scale values are between 1-9, so we have to normalise the scale to [-1,1] range
# this is as per Russel's Circumplex Alignment
df_targets["valence_norm"] = (df_targets["valence_mean"] - 5.0) / 4.0
df_targets["arousal_norm"] = (df_targets["arousal_mean"] - 5.0) / 4.0

print(f"Loaded {len(df_targets)} target labels")




Loaded 1744 target labels


LOAD AND AGGREGATE OPENSMILE FEATURES

openSMILE- open source Speech and Music Interpretation by large space extraction is used to extract acoustic features from  audio signals

Merge features and targets, create train test split

In [5]:

all_csv = glob.glob(os.path.join(dataset_dir, "**", "*.csv"), recursive=True)
features_files = [
    f for f in all_csv if os.path.basename(f).replace(".csv", "").isdigit()
]

print(f"Found {len(features_files)} feature files. Aggregating...")

agg_features = []
for file in features_files:
  try:
    song_id = int(os.path.basename(file).split(".")[0])
    df_feat = pd.read_csv(file, sep=";")

    if df_feat.empty:
      continue

    num_cols = df_feat.select_dtypes(include=[np.number]).columns
    num_cols = [
        c
        for c in num_cols
        if c.lower() not in ["frametime", "frame_time", "time"]
    ]

    if not num_cols:
        continue

    m = df_feat[num_cols].mean()
    s = df_feat[num_cols].std()

    row_dict = {"song_id": song_id}
    for col in num_cols:
      row_dict[f"{col}_mean"] = m[col] if pd.notna(m[col]) else 0.0
      row_dict[f"{col}_std"] = s[col] if pd.notna(s[col]) else 0.0

    agg_features.append(row_dict)
  except Exception as e:
    print(f"Error processing file {file}: {e}")
    continue

df_features = pd.DataFrame(agg_features)
df_features.drop_duplicates(subset=["song_id"], keep="first", inplace=True)

print(
    f"Successfully aggregated {df_features.shape[1] - 1} features across"
    f" {len(df_features)} unique tracks."
)


Found 5406 feature files. Aggregating...
Successfully aggregated 520 features across 1802 unique tracks.


In [6]:
df_dataset = pd.merge(df_targets, df_features, on="song_id", how="inner")
print(f"Dataset shape: {df_dataset.shape}")

Dataset shape: (1744, 525)


Here, I am removing zero variance and highly correlated features (i.e., |r|>0.95) to reduce dimensionality and multicolllinearity before model training. Train test split is done before feautre selection stats were computed to avoid information leakage from test set


In [8]:
metadata_cols = ["song_id", "valence_mean", "arousal_mean", "valence_norm", "arousal_norm"]
feature_cols = [c for c in df_dataset.columns if c not in metadata_cols]

X = df_dataset[feature_cols].copy()
y = df_dataset[["valence_norm", "arousal_norm"]].copy()

X = X.astype(float)
X.fillna(X.median(), inplace = True)

print(f"Sum of S.D after fillna: {X.std().sum()}")
print(f"No. of columns with std>0: {(X.std()>0).sum()}")

#train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#drop zero variance features, fit on train set
train_std = X_train.std()
zero_var_cols = train_std[train_std == 0].index
X_train = X_train.drop(columns=zero_var_cols)
X_test = X_test.drop(columns=zero_var_cols)

#fit correlation pruning
corr_matrix = X_train.corr().abs().values
num_features = X_train.shape[1]
to_drop = set()
for i in range(num_features):
  if i in to_drop:
    continue
  for j in range(i+1, num_features):
    if corr_matrix[i,j] >= 0.95:
      to_drop.add(j)
cols_to_drop = [X_train.columns[i] for i in to_drop]
X_train = X_train.drop(columns=cols_to_drop)
X_test = X_test.drop(columns=cols_to_drop)

print(f"Features remaining: {X_train.shape[1]}")
#scale features

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)
# Save Processed Matrices
X_train_scaled.to_csv(os.path.join(OUTPUT_PATH, "X_train.csv"), index=False)
X_test_scaled.to_csv(os.path.join(OUTPUT_PATH, "X_test.csv"), index=False)
y_train.to_csv(os.path.join(OUTPUT_PATH, "y_train.csv"), index=False)
y_test.to_csv(os.path.join(OUTPUT_PATH, "y_test.csv"), index=False)

print("\n--- DATA PREPROCESSING COMPLETED ---")
print(f"Training set: {X_train_scaled.shape[0]} samples | Test set: {X_test_scaled.shape[0]} samples")
print(f"Features ready for ML model training: {X_train_scaled.shape[1]}")

Sum of S.D after fillna: 5034622.389442427
No. of columns with std>0: 520
Features remaining: 302

--- DATA PREPROCESSING COMPLETED ---
Training set: 1395 samples | Test set: 349 samples
Features ready for ML model training: 302


In [11]:
OUTPUT_PATH = "/content/processed_data"

print("Checking directory contents:")
if os.path.exists(OUTPUT_PATH):
  print("Files found in OUTPUT_PATH:", os.listdir(OUTPUT_PATH))
else:
  print("OUTPUT_PATH directory does not exist yet!")

Checking directory contents:
Files found in OUTPUT_PATH: ['X_train.csv', 'X_test.csv', 'y_train.csv', 'y_test.csv']


In [12]:
drive.mount('/content/drive')

DRIVE_PATH = "/content/drive/MyDrive/processed_data"
os.makedirs(DRIVE_PATH, exist_ok=True)

X_train_scaled.to_csv(os.path.join(DRIVE_PATH, "X_train.csv"), index=False)
X_test_scaled.to_csv(os.path.join(DRIVE_PATH, "X_test.csv"), index=False)
y_train.to_csv(os.path.join(DRIVE_PATH, "y_train.csv"), index=False)
y_test.to_csv(os.path.join(DRIVE_PATH, "y_test.csv"), index=False)

Mounted at /content/drive


### NOTEBOOK SUMMARY
- 1744 DEAM tracks merged with openSMILE features

- train test split

- variance filtering

- Correlation pruning

- Final feature set: 302 acoustic features across 1,395 training / 349 test samples